# 04 · Offline ablation — re-rank from stored score components, no API calls

**No real `01-modules/01-tools/04-retrieve` output exists yet to read — every score below is a synthetic stand-in, built in the exact shape stage-04 is expected to write.**

The idea: once a paper has been scored, its **components** are stored alongside the final score:

```
final_score = w_rel * relevance + w_cit * citations + w_rec * recency
            + w_jq * journal_impact + w_ev * evidence_level
```

If the five components are stored per-paper, the ranking can be
**recomputed with different weights against the exact same components** —
no re-retrieval, no re-scoring by an LLM, no API call, no cost, and no
run-to-run variance. This is the cheapest useful experiment in this whole
repo, and the best first thing for a contributor with no API key.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `rerank` | Recomputes `final_score` from stored `final_score_components` under a given set of weights — a pure function, no API call | `rerank(scored_papers, weights)` |
| `print_ranking` | Prints a ranked list of papers with their final score and evidence level | `print_ranking(baseline, "baseline weights")` |
| `adjust_weight` | Changes ONE weight in a weights dict by a delta — the building block the full ablation is assembled from | `adjust_weight(weights, "evidence_level", 0.15)` |


## Step 1 — locate the repo root and confirm the environment

This notebook lives two levels below the repo root, so the first thing it does is walk up the directory tree to find `nbio.py` and import it — everything else in this notebook depends on `repo_root` being set correctly.

In [ ]:
# This notebook lives two levels below the repo root (01-modules/01-tools/06-bench/),
# and Jupyter starts a kernel with its working directory set to the
# notebook's own folder -- so nbio.py (at the repo root) is not importable
# yet. Walk up until we find it, same logic nbio.bootstrap() uses
# internally once it CAN be imported.
import sys
from pathlib import Path

def _find_repo_root(start):
    root = start.resolve()
    for _ in range(6):
        if (root / "nbio.py").is_file():
            return root
        root = root.parent
    raise RuntimeError("could not locate nbio.py above the current directory")

_repo_root_for_import = _find_repo_root(Path.cwd())
if str(_repo_root_for_import) not in sys.path:
    sys.path.insert(0, str(_repo_root_for_import))

import nbio
repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 2 — the stored components this notebook works from

Each entry below is what a *real* run would have written per retrieved
paper: five normalised signals in `[0, 1]`, plus enough metadata to display
it. **No `04-retrieve` output exists to read yet at the time this notebook
was written** (that stage is being built separately, in parallel) — this
notebook builds a small synthetic set of "stored components" in the same
shape rather than block on it. In a real stage-04 output this would be read
from `runs/<run_id>/papers.json` via `nbio.load_run()` — see the note at the
end of this notebook.

In [ ]:
# SYNTHETIC -- stands in for real papers[].final_score_components once
# 01-modules/01-tools/04-retrieve writes that shape to runs/<run_id>/papers.json.
scored_papers = [
    {
        "title": "Early excision timing in deep partial thickness burns (meta-analysis)",
        "evidence_level": "Level I",
        "final_score_components": {
            "relevance": 0.80, "citations": 0.70, "recency": 0.30,
            "journal_impact": 0.80, "evidence_level": 1.00,
        },
    },
    {
        "title": "Case series: skin grafting outcomes at a single burn center",
        "evidence_level": "Level IV",
        "final_score_components": {
            "relevance": 0.95, "citations": 0.20, "recency": 0.95,
            "journal_impact": 0.30, "evidence_level": 0.50,
        },
    },
    {
        "title": "Expert opinion: grafting timing in resource-limited settings",
        "evidence_level": "Level V",
        "final_score_components": {
            "relevance": 0.60, "citations": 0.10, "recency": 0.20,
            "journal_impact": 0.20, "evidence_level": 0.25,
        },
    },
]

print(f"{len(scored_papers)} synthetic papers, each with 5 stored score components")
for p in scored_papers:
    print(f"  {p['evidence_level']:<10} {p['title'][:60]}")

## Step 3 — the scoring weights this run uses

Department scoring weights, in the same shape as `corpus.yaml`'s
`scoring_weights` in the source product. Deliberately light on
`evidence_level` here — enough that, as the next steps show, a paper with
weaker evidence but more relevance and recency can outrank a stronger
meta-analysis under these weights.

In [ ]:
weights = {
    "relevance": 0.45,
    "citations": 0.15,
    "recency": 0.25,
    "journal_impact": 0.10,
    "evidence_level": 0.05,
}
assert abs(sum(weights.values()) - 1.0) < 1e-9, "weights should sum to 1.0"
print("weights:", weights)

## Step 4 — the ranking function: a pure function of stored components

Same components in, same order and scores out, every time — no model call,
no clock, no randomness. This determinism is what makes a ranking auditable
after the fact: given the stored components and the weights that were
active, anyone can re-derive exactly why a paper landed where it did.

In [ ]:
def rerank(papers, weights):
    def score(p):
        c = p["final_score_components"]
        return sum(weights[k] * c[k] for k in weights)

    scored = [{**p, "final_score": round(score(p), 4)} for p in papers]
    return sorted(scored, key=lambda p: p["final_score"], reverse=True)

## Step 5 — a print helper, then rank the papers under the baseline weights

In [ ]:
def print_ranking(ranked, title):
    print(f"-- {title} --")
    for i, p in enumerate(ranked, start=1):
        print(f"  {i}. final={p['final_score']:.4f}  {p['evidence_level']:<10} {p['title'][:56]}")

In [ ]:
baseline = rerank(scored_papers, weights)
print_ranking(baseline, "baseline weights")

## Step 6 — prove the ranking is deterministic before trusting it

Same components, same weights, run twice — the order and the scores must
match exactly, or nothing downstream that assumes reproducibility can be
trusted.

In [ ]:
again = rerank(scored_papers, weights)
same_order = [p["title"] for p in baseline] == [p["title"] for p in again]
same_scores = all(abs(a["final_score"] - b["final_score"]) < 1e-12 for a, b in zip(baseline, again))
print(f"deterministic: same order={same_order}, same scores={same_scores}")
assert same_order and same_scores

## Step 7 — build a one-weight-change helper, test it on a single component

This is claim **C4** from the private lab's paper plan (grading evidence by
Oxford CEBM level improves the final ranking), approached one weight at a
time instead of hand-editing the whole dict at once. `adjust_weight` changes
exactly one weight by a delta — tested here on `evidence_level` alone, before
it's used to build the full ablation. Note the weights won't sum to 1.0
after this single change; that's expected and gets fixed in the next
step.

In [ ]:
def adjust_weight(weights, key, delta):
    import copy
    new_weights = copy.deepcopy(weights)
    new_weights[key] += delta
    return new_weights

In [ ]:
adjusted_once = adjust_weight(weights, "evidence_level", 0.15)
print("evidence_level: before =", weights["evidence_level"], " after =", adjusted_once["evidence_level"])
print("sum of weights after one change:", round(sum(adjusted_once.values()), 4), "(not 1.0 yet -- expected)")

## Step 8 — apply the helper three times to build the full ablation

Take weight away from `citations` and `recency`, give it all to
`evidence_level` — the same net change as before, now built up from the
tested `adjust_weight` helper instead of edited on the dict directly.

In [ ]:
heavier_evidence_weights = adjust_weight(weights, "citations", -0.05)
heavier_evidence_weights = adjust_weight(heavier_evidence_weights, "recency", -0.10)
heavier_evidence_weights = adjust_weight(heavier_evidence_weights, "evidence_level", 0.15)
assert abs(sum(heavier_evidence_weights.values()) - 1.0) < 1e-9
print("evidence-heavy weights:", heavier_evidence_weights)

## Step 9 — re-rank under the new weights and see whether the order moves

Nothing about the papers changes — only the weight. If `evidence_level`'s
weight goes up, evidence-graded papers should rise and thin case-series/
opinion papers should fall, using the exact same underlying scores.

In [ ]:
reranked = rerank(scored_papers, heavier_evidence_weights)
print_ranking(reranked, "evidence-heavy weights")

before_order = [p["title"] for p in baseline]
after_order = [p["title"] for p in reranked]
print("\norder changed:", before_order != after_order)
for title in before_order:
    old_pos = before_order.index(title) + 1
    new_pos = after_order.index(title) + 1
    moved = "" if old_pos == new_pos else f"  ({'up' if new_pos < old_pos else 'down'} {abs(old_pos - new_pos)})"
    print(f"  {title[:56]:<58} {old_pos} -> {new_pos}{moved}")

## Step 10 — re-derive the top score by hand

If this assertion holds, the stored components fully explain the stored
score — the property that makes a ranking auditable months after the run
that produced it, without re-calling any model.

In [ ]:
top = reranked[0]
c = top["final_score_components"]
by_hand = sum(heavier_evidence_weights[k] * c[k] for k in heavier_evidence_weights)
print(f"stored final_score : {top['final_score']:.6f}")
print(f"recomputed by hand : {by_hand:.6f}")
assert abs(by_hand - top["final_score"]) < 1e-9, "components do not explain the score"
print("\ncomponents fully explain the score")

## Reading real stage-04 output, once it exists

Once `01-modules/01-tools/04-retrieve` writes `runs/<run_id>/papers.json` with a
`final_score_components` field per paper, this notebook's `scored_papers`
becomes:

```python
run = nbio.load_run()  # most recent run, or nbio.load_run("<run_id>")
scored_papers = run["papers"]
```

...and every cell above runs unchanged against real retrieval output instead
of the synthetic set. That's the point of `final_score_components` living on
the paper record rather than only the final number: this notebook's ablation
is not specific to synthetic data, only currently limited to it.